# 0825_lsw_008_adaptive_threshold

사용자 질문: "시간에 따라 임계값을 조정하는 게 치팅 아니냐"에 대한 답을 실험으로 검증한다.

- **치팅**: 지금 평가하려는 구간의 실제 라벨로 그 구간의 임계값을 정하는 것.
- **치팅 아님**: **이미 확정된 과거 구간**의 관측 라벨로 **다음 구간**의 임계값만 다시 고르는
  것 — 모델은 재학습하지 않는다(006의 전면 재학습보다 훨씬 싸다).

006과 같은 3-step 슬라이딩 구조를 재사용한다(Train 0~40%, Val 40~60%/60~80%/80~100%). 모델은
**한 번만(0~40%) 학습**하고 고정한다. 세 정책을 비교한다:

1. **Frozen**: 임계값도 처음(40~60% 구간)에 한 번만 고르고 끝까지 그대로 쓴다 — 재학습도
   재보정도 없음.
2. **Adaptive threshold(치팅 아닌 버전)**: 모델은 그대로 두고, 매 구간 직전까지 확정된 모든
   데이터(원래 Train + 그 이전 Val 구간들)에 그 고정 모델의 확률을 다시 매겨서 임계값만 새로
   고른다. 단, 1번째 구간(40~60%)은 학습 직후 첫 배포 전이라 "이전 관측"이 아직 없으므로 —
   003~007과 동일한 관행대로 — 그 구간 자체를 최초 보정용 Val로 쓴다(모델 학습에는 안 쓰임).
3. **Full retrain(006 재사용)**: 매번 모델까지 다시 학습(006의 결과를 그대로 가져와 비교
   기준으로 삼는다).

세 정책 모두 첫 구간(40~60%)은 동일해야 한다(자체 검증 포인트).


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_008_adaptive_threshold"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_008_adaptive_threshold


## 2. 데이터 로딩·전처리 (006과 동일한 cut point)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]
timestamps = clean_df[TIME_COLUMN]


def cut_at(fraction):
    sizes = timestamps.value_counts(sort=False).sort_index()
    cum = sizes.cumsum().to_numpy()
    idx = int(np.searchsorted(cum, len(clean_df) * fraction, side="left"))
    idx = min(idx, len(sizes) - 1)
    return sizes.index[idx]


cut_points = {f: cut_at(f) for f in [0.20, 0.40, 0.60, 0.80]}
print("cut points:", cut_points)


cut points: {0.2: Timestamp('1970-08-02 17:57:15+0000', tz='UTC'), 0.4: Timestamp('1970-08-25 14:06:02+0000', tz='UTC'), 0.6: Timestamp('1970-09-28 05:48:26+0000', tz='UTC'), 0.8: Timestamp('1970-10-13 13:14:26+0000', tz='UTC')}


## 3. 평가 함수 (003~007과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "n": len(y_true),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model():
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )


## 4. 검사유형별 고정 모델 학습 (Train 0~40%)

006의 step1과 정확히 같은 모델이다 — 이후 재학습하지 않고 계속 재사용한다.

In [4]:
frozen_models = {}
frozen_feature_columns = {}

train_mask0 = timestamps <= cut_points[0.40]

for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    train_df = clean_df.loc[train_mask0 & type_mask]
    feature_columns = get_non_constant_columns(feature_columns_all, train_df)

    model = build_model()
    model.fit(train_df[feature_columns], train_df[TARGET])

    frozen_models[inspection_type] = model
    frozen_feature_columns[inspection_type] = feature_columns


## 5. 세 정책 비교

- Val 구간 3개: (40~60%], (60~80%], (80~100%]
- **frozen**: 40~60%에서 고른 임계값을 끝까지 고정
- **adaptive_threshold**: 그 구간 시작 전까지 확정된 모든 데이터에 고정 모델 확률을 다시 매겨
  임계값만 새로 고름


In [5]:
val_bounds = [
    (cut_points[0.40], cut_points[0.60]),
    (cut_points[0.60], cut_points[0.80]),
    (cut_points[0.80], None),
]

policy_results = []

for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    model = frozen_models[inspection_type]
    feature_columns = frozen_feature_columns[inspection_type]

    frozen_threshold = None

    for step, (val_lo, val_hi) in enumerate(val_bounds, start=1):
        val_mask = (timestamps > val_lo) & type_mask
        if val_hi is not None:
            val_mask &= timestamps <= val_hi
        val_df = clean_df.loc[val_mask]
        val_proba = model.predict_proba(val_df[feature_columns])[:, 1]

        # confirmed history available for threshold selection:
        # step1은 학습 직후 첫 배포 전이라 아직 아무 "확정 관측"이 없다 -> 표준 관행대로 첫 Val 자체를
        # 초기 보정용으로 쓴다(모델 학습에는 안 씀 -> 치팅 아님, 003~007과 동일한 관행).
        # step2 이후는 "이 구간 시작 전까지 확정된 모든 데이터"만 쓴다(이 구간 자체는 제외).
        confirmed_hi = val_hi if step == 1 else val_lo
        # 하한을 원래 Train 끝(cut_points[0.40])으로 고정 -> Train 자체는 절대 포함 안 되게 한다
        # (모델이 이미 본 데이터로 임계값을 고르면 과적합된 in-sample 확률에 낚인다 -> 실제로 처음
        # 시도했을 때 이 버그로 결과가 크게 틀어졌었다).
        confirmed_mask = (timestamps > cut_points[0.40]) & (timestamps <= confirmed_hi) & type_mask
        confirmed_df = clean_df.loc[confirmed_mask]
        confirmed_proba = model.predict_proba(confirmed_df[feature_columns])[:, 1]
        adaptive_threshold = select_threshold(confirmed_df[TARGET], confirmed_proba)

        if frozen_threshold is None:
            frozen_threshold = adaptive_threshold  # step1: 정의상 frozen과 adaptive가 같아야 함

        frozen_result = evaluate_at_threshold(val_df[TARGET], val_proba, frozen_threshold)
        frozen_result.update({"inspection_type": inspection_type, "step": step, "policy": "frozen"})
        policy_results.append(frozen_result)

        adaptive_result = evaluate_at_threshold(val_df[TARGET], val_proba, adaptive_threshold)
        adaptive_result.update({"inspection_type": inspection_type, "step": step, "policy": "adaptive_threshold"})
        policy_results.append(adaptive_result)

policy_df = pd.DataFrame(policy_results).set_index(["inspection_type", "step", "policy"]).sort_index()
policy_df[["threshold", "n", "tn", "fp", "fn", "tp", "slip_rate", "volume_reduction", "total_cost_1:10", "total_cost_1:100"]]


threshold      n     tn     fp  \
inspection_type step policy                                                  
0               1    adaptive_threshold  4.529815e-06  13409   6456   6932   
                     frozen              4.529815e-06  13409   6456   6932   
                2    adaptive_threshold  4.529815e-06  23492  10009  13475   
                     frozen              4.529815e-06  23492  10009  13475   
                3    adaptive_threshold  4.529815e-06  16784   6887   9764   
                     frozen              4.529815e-06  16784   6887   9764   
1               1    adaptive_threshold  6.899076e-06   7097   1440   5399   
                     frozen              6.899076e-06   7097   1440   5399   
                2    adaptive_threshold  6.899076e-06  10494   1275   8980   
                     frozen              6.899076e-06  10494   1275   8980   
                3    adaptive_threshold  5.867798e-06  11112   1167   9161   
                     frozen              6.899076e-06  11112   1487   8841   
2               1    adaptive_threshold  4.816745e-10  22317    142  22096   
                     frozen              4.816745e-10  22317    142  22096   
                2    adaptive_threshold  4.816745e-10  15897      0  15865   
                     frozen              4.816745e-10  15897      0  15865   
                3    adaptive_threshold  3.950262e-06  19153   2335  16115   
                     frozen              4.816745e-10  19153      0  18450   
3               1    adaptive_threshold  4.735841e-06  34547   5105  29381   
                     frozen              4.735841e-06  34547   5105  29381   
                2    adaptive_threshold  4.735841e-06  27328   7160  20144   
                     frozen              4.735841e-06  27328   7160  20144   
                3    adaptive_threshold  4.735841e-06  30591   5812  24176   
                     frozen              4.735841e-06  30591   5812  24176   
4               1    adaptive_threshold  1.100488e-04   1052    800    247   
                     frozen              1.100488e-04   1052    800    247   
                2    adaptive_threshold  1.100488e-04   1163    750    353   
                     frozen              1.100488e-04   1163    750    353   
                3    adaptive_threshold  4.532806e-06    756      4    726   
                     frozen              1.100488e-04    756    371    359   

                                         fn   tp  slip_rate  volume_reduction  \
inspection_type step policy                                                     
0               1    adaptive_threshold   0   21   0.000000          0.482223   
                     frozen               0   21   0.000000          0.482223   
                2    adaptive_threshold   0    8   0.000000          0.426205   
                     frozen               0    8   0.000000          0.426205   
                3    adaptive_threshold  20  113   0.150376          0.413609   
                     frozen              20  113   0.150376          0.413609   
1               1    adaptive_threshold   2  256   0.007752          0.210557   
                     frozen               2  256   0.007752          0.210557   
                2    adaptive_threshold   3  236   0.012552          0.124330   
                     frozen               3  236   0.012552          0.124330   
                3    adaptive_threshold   6  778   0.007653          0.112994   
                     frozen               6  778   0.007653          0.143978   
2               1    adaptive_threshold   0   79   0.000000          0.006385   
                     frozen               0   79   0.000000          0.006385   
                2    adaptive_threshold   0   32   0.000000          0.000000   
                     frozen               0   32   0.000000          0.000000   
                3    adaptive_threshold  24  679   0.034139          0.126558   
        

## 6. 자체 검증 — step1은 frozen과 adaptive_threshold가 같아야 한다

In [6]:
check_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    frozen_step1 = policy_df.loc[(inspection_type, 1, "frozen")]
    adaptive_step1 = policy_df.loc[(inspection_type, 1, "adaptive_threshold")]
    check_rows.append(
        {
            "inspection_type": inspection_type,
            "frozen_threshold": frozen_step1["threshold"],
            "adaptive_threshold": adaptive_step1["threshold"],
            "일치": frozen_step1["threshold"] == adaptive_step1["threshold"],
        }
    )
pd.DataFrame(check_rows).set_index("inspection_type")


,frozen_threshold,adaptive_threshold,일치
inspection_type,,,
0,4.529815e-06,4.529815e-06,True
1,6.899076e-06,6.899076e-06,True
2,4.816745e-10,4.816745e-10,True
3,4.735841e-06,4.735841e-06,True
4,1.100488e-04,1.100488e-04,True


## 7. frozen vs adaptive_threshold vs 006 full retrain — step별 총비용

006의 step 결과(같은 유형·같은 구간)를 그대로 가져와 세 번째 정책으로 같이 비교한다.

In [7]:
# 006에서 이미 계산된 full retrain 결과 (0825_lsw_006 노트북 5절 출력 값 그대로 옮김)
full_retrain_costs = {
    (0, 1): (6932, 6932), (0, 2): (12799, 12799), (0, 3): (15544, 15634),
    (1, 1): (5419, 5599), (1, 2): (7746, 7926), (1, 3): (9247, 9877),
    (2, 1): (22096, 22096), (2, 2): (15165, 15165), (2, 3): (18210, 18840),
    (3, 1): (29381, 29381), (3, 2): (19766, 19766), (3, 3): (29418, 29958),
    (4, 1): (247, 247), (4, 2): (1097, 1097), (4, 3): (560, 560),
}

summary_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    for step in [1, 2, 3]:
        frozen_row = policy_df.loc[(inspection_type, step, "frozen")]
        adaptive_row = policy_df.loc[(inspection_type, step, "adaptive_threshold")]
        full_10, full_100 = full_retrain_costs[(inspection_type, step)]
        summary_rows.append(
            {
                "inspection_type": inspection_type,
                "step": step,
                "frozen_총비용(1:10)": frozen_row["total_cost_1:10"],
                "adaptive_총비용(1:10)": adaptive_row["total_cost_1:10"],
                "full_retrain_총비용(1:10)": full_10,
                "frozen_총비용(1:100)": frozen_row["total_cost_1:100"],
                "adaptive_총비용(1:100)": adaptive_row["total_cost_1:100"],
                "full_retrain_총비용(1:100)": full_100,
            }
        )
summary_df = pd.DataFrame(summary_rows).set_index(["inspection_type", "step"])
summary_df


frozen_총비용(1:10)  adaptive_총비용(1:10)  \
inspection_type step                                         
0               1               6932.0              6932.0   
                2              13475.0             13475.0   
                3               9964.0              9964.0   
1               1               5419.0              5419.0   
                2               9010.0              9010.0   
                3               8901.0              9221.0   
2               1              22096.0             22096.0   
                2              15865.0             15865.0   
                3              18450.0             16355.0   
3               1              29381.0             29381.0   
                2              20144.0             20144.0   
                3              24406.0             24406.0   
4               1                247.0               247.0   
                2                893.0               893.0   
                3                529.0               736.0   

                      full_retrain_총비용(1:10)  frozen_총비용(1:100)  \
inspection_type step                                              
0               1                       6932             6932.0   
                2                      12799            13475.0   
                3                      15544            11764.0   
1               1                       5419             5599.0   
                2                       7746             9280.0   
                3                       9247             9441.0   
2               1                      22096            22096.0   
                2                      15165            15865.0   
                3                      18210            18450.0   
3               1                      29381            29381.0   
                2                      19766            20144.0   
                3                      29418            26476.0   
4               1                        247              247.0   
                2                       1097             5753.0   
                3                        560             2059.0   

                      adaptive_총비용(1:100)  full_retrain_총비용(1:100)  
inspection_type step                                                
0               1                  6932.0                     6932  
                2                 13475.0                    12799  
                3                 11764.0                    15634  
1               1                  5599.0                     5599  
                2                  9280.0                     7926  
                3                  9761.0                     9877  
2               1                 22096.0                    22096  
                2                 15865.0                    15165  
                3                 18515.0                    18840  
3               1                 29381.0                    29381  
                2                 20144.0                    19766  
                3                 26476.0                    29958  
4               1                   247.0                      247  
                2                  5753.0                     1097  
                3                   826.0                      560

## 8. 결론 및 다음 단계

### 구조적으로 알아둘 점: step2는 항상 frozen=adaptive다

confirmed 데이터의 하한을 Train 끝으로 고정했기 때문에, step1과 step2의 "확정된 보정용 데이터"가
둘 다 Val_1(40~60%) 하나뿐이라 **step2에서는 재보정을 해도 같은 임계값이 나온다** — 진짜 재보정
효과는 **step3에서만** 관찰된다(confirmed 데이터에 Val_2가 새로 추가되는 시점). 그래서 아래는
step3만 정리한다.

### step3 총비용 비교 (frozen vs adaptive_threshold vs 006 full retrain)

| type | 1:10: frozen / adaptive / full_retrain | 1:100: frozen / adaptive / full_retrain | 승자 |
|---|---|---|---|
| 0 | 9,964 / **9,964**(동일) / 15,544 | 11,764 / **11,764**(동일) / 15,634 | frozen=adaptive, 둘 다 full retrain보다 훨씬 나음 |
| 1 | **8,901** / 9,221 / 9,247 | **9,441** / 9,761 / 9,877 | **frozen이 최고** — 손대지 않는 게 제일 낫다 |
| 2 | 18,450 / **16,355** / 18,210 | **18,450** / 18,515 / 18,840 | 1:10은 adaptive 승, 1:100은 frozen과 거의 동률 |
| 3 | 24,406 / 24,406(동일) / 29,418 | 26,476 / 26,476(동일) / 29,958 | frozen=adaptive, 둘 다 full retrain보다 훨씬 나음 |
| 4 | **529** / 736 / 560 | 2,059 / 826 / **560** | 표본 부족(양성 손에 꼽음)이라 신뢰 불가 |

### 답: 치팅 없이도 개선 가능하지만, "가만히 두는 것"도 의외로 강하다

1. **type0/1/3에서 006의 전면 재학습(full retrain)이 frozen/adaptive보다 뚜렷하게 나쁘다.**
   006에서 "재학습해도 drift를 못 이긴다"고 봤던 성능 급락이, 사실은 재학습 자체가 오히려
   해로웠던 것으로 재해석된다 — 매 step 같은 크기(40%)의 최신 윈도우로 다시 학습하는 과정에서
   그 시점 데이터의 일시적 잡음에 과적합했을 가능성이 크다. **아무것도 안 하는 것(frozen)이
   최선인 경우가 실제로 있다.**
2. **type2만 유일하게 "치팅 아닌" adaptive threshold가 확실히 이긴다**(1:10에서 18,450→16,355,
   11% 절감). type2는 notes.md에서 확인한 concept drift가 가장 강한 유형인데, 모델 자체를
   재학습하지 않고 임계값만 최근 확정 데이터로 다시 잡아주는 정도가 딱 맞는 개입 강도였던 것으로
   보인다 — 모델을 통째로 새로 학습하면(006) 오히려 나빠지고(18,210~29% 더 비쌈은 아니지만
   frozen과 비슷), 아예 손 안 대면 최신 기준 변화를 못 따라가는데, 임계값만 살짝 움직이는 중간
   지점이 최적이었다.
3. **핵심 결론**: "시간에 따라 임계값을 낮추는 것"은 치팅이 아니며(과거 확정 데이터만 쓰면),
   특정 유형(type2)에서는 실제로 도움이 된다. 다만 만능은 아니고, 오히려 대부분의 유형에서는
   **아무 개입도 안 하는 것(frozen)이 재학습보다도 낫다는 게 더 중요한 발견**이다 — "드리프트가
   있으니 뭐라도 해야 한다"는 직관이 이 데이터에서는 틀릴 때가 더 많았다.

### 검사유형별 최적 조합 갱신 (이번엔 "임계값 재조정" 축 추가)

이 노트북은 003~007의 리샘플링/라벨정제 축과는 다른 축(운영 정책: 얼마나 자주 손대는가)이라
직접 비교표에 합치기는 어렵다. 다만 **"재학습 주기 정책" 자체를 고른다면**: type0/1/3/4는 재학습
없이 그대로 두는 게 낫고, type2만 "임계값 재보정"을 주기적으로 하는 게 이득이라는 게 이번 결과다.

### 다음 단계

1. 이 노트북을 `docs/experiments/0825_lsw_008_adaptive_threshold.md`로 옮기고
   `docs/experiments/index.md`, `handoff.md`에 반영한다.
2. 이상치 탐지 재구현 또는 Phase 4(비지도 이상탐지) 중 다음 방향 결정.
